In [24]:
# ============================================================================
# DATA QUALITY ASSESSMENT - WARHAMMER 40K TOURNAMENT DATABASE
# ============================================================================
# This notebook evaluates data quality across multiple dimensions:
# - Completeness, Accuracy, Consistency, Uniqueness, Validity
# Comparing data BEFORE and AFTER integration/enrichment phases
# ============================================================================

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime


# Import custom data loading functions
from Warhammer_SQL import (
    load_tournament_data,
    create_army_lists_df,
    explode_army_lists,
    load_prices
)

### Configuration

In [25]:
# Configuration
JSON_FILE = "warhammer_final_cleaned.json"
PRICES_FILE = "Warhammer_prezzi.csv"
DB_NAME = "warhammer_tournaments.db"

# Load raw tournament data (BEFORE enrichment)
df_tournament, extended_lists = load_tournament_data(JSON_FILE)
df_army_lists = create_army_lists_df(extended_lists)
df_expanded_before = explode_army_lists(df_army_lists)
df_prices, df_prices_raw, df_prices_army = load_prices(PRICES_FILE)

print(f"✓ Tournament: {df_tournament['NAME'].iloc[0]}")
print(f"✓ Players: {len(extended_lists)}")
print(f"✓ Units before enrichment: {len(df_expanded_before)}")

✓ Tournament: Warhammer 40,000 - 2025 World Championships of Warhammer
✓ Players: 402
✓ Units before enrichment: 4584


In [26]:
# Load enriched data from database (AFTER enrichment)
conn = sqlite3.connect(DB_NAME)

df_tournaments_after = pd.read_sql("SELECT * FROM tournaments", conn)
df_players_after = pd.read_sql("SELECT * FROM players", conn)
df_army_units_after = pd.read_sql("SELECT * FROM army_list_units", conn)
df_matches_after = pd.read_sql("SELECT * FROM matches", conn)
df_army_lists_after = pd.read_sql("SELECT * FROM army_lists", conn)

print(f"✓ Database loaded: {DB_NAME}")
print(f"✓ Units after enrichment: {len(df_army_units_after)}")

✓ Database loaded: warhammer_tournaments.db
✓ Units after enrichment: 4584


### 1. Reference Data Redundancy (No Duplicates)

#### Reference Data Redundancy in Price Reference Data

In [27]:

df_prices, df_prices_raw, df_prices_army = load_prices(PRICES_FILE)

def assess_redundancy_prices(df_raw, df_clean):
    """Assess duplicate records in price reference data"""
    
    dup_before = len(df_raw) - len(df_raw[['clean_name', 'points_unit', 'PRICE_unit']].drop_duplicates())
    dup_after = df_clean.duplicated().sum()
    
    results = pd.DataFrame({
        "Stage": ["Before Cleaning", "After Cleaning"],
        "Total Records": [len(df_raw), len(df_clean)],
        "Duplicate Records": [dup_before, dup_after],
        "Redundancy Rate (%)": [
            (dup_before / len(df_raw)) * 100,
            (dup_after / len(df_clean)) * 100
        ]
    })
    
    return results

redundancy_prices = assess_redundancy_prices(df_prices_army, df_prices)
print("REDUNDANCY - Price Reference Data:")
print(redundancy_prices.to_string(index=False))
print(f"\n✓ Improvement: {redundancy_prices.iloc[0]['Duplicate Records']} duplicates removed")


REDUNDANCY - Price Reference Data:
          Stage  Total Records  Duplicate Records  Redundancy Rate (%)
Before Cleaning           1386                550             39.68254
 After Cleaning            836                  0              0.00000

✓ Improvement: 550 duplicates removed


#### Instance-level Uniqueness in Match Records

In [28]:

def assess_uniqueness_matches(extended_lists, df_players_after, conn):
    """Assess duplicate match records (each match appears twice in raw data)"""
    
    # Count raw match records (each match recorded from both players' perspectives)
    total_raw = sum(len(player.get("path", [])) for player in extended_lists)
    expected_unique = (df_players_after['WINS'].sum() + df_players_after['LOSSES'].sum() + df_players_after['TIES'].sum())//2


    
    # Check for duplicates in cleaned database
    query = """
        SELECT COUNT(*) as duplicates
        FROM (
            SELECT ID_TOURNAMENT, ID_PLAYER1, ID_PLAYER2, SCORE_PLAYER1, SCORE_PLAYER2
            FROM matches
            GROUP BY ID_TOURNAMENT, ID_PLAYER1, ID_PLAYER2, SCORE_PLAYER1, SCORE_PLAYER2
            HAVING COUNT(*) > 1
        )
    """
    dup_after = pd.read_sql(query, conn)['duplicates'].iloc[0]
    
    results = pd.DataFrame({
        "Stage": ["Before Deduplication", "After Deduplication"],
        "Match Records": [total_raw, expected_unique],
        "Expected Unique": [total_raw//2, expected_unique],
        "Duplicates": [total_raw - expected_unique, dup_after],
        "Redundancy Rate": [0.5, dup_after / expected_unique if expected_unique > 0 else 0]
    })
    
    return results

uniqueness_matches = assess_uniqueness_matches(extended_lists, df_players_after, conn)
print("\nUNIQUENESS - Match Records:")
print(uniqueness_matches.to_string(index=False))
print(f"\n✓ {uniqueness_matches.iloc[0]['Duplicates']} duplicate matches removed")


UNIQUENESS - Match Records:
               Stage  Match Records  Expected Unique  Duplicates  Redundancy Rate
Before Deduplication           3206             1603        1603              0.5
 After Deduplication           1603             1603           0              0.0

✓ 1603 duplicate matches removed


### 2. Name Matching ACCURACY

In [29]:
 
# Evaluates the quality of name normalization (blank spaces removing) and matching between datasets.

def assess_syntactic_accuracy(df_expanded_before, df_prices_raw, df_army_units_after, df_prices):
    """Compare unit name matching before and after normalization"""
    
    # BEFORE normalization
    raw_unit_names = set(df_expanded_before['clean_name'].str.lower().unique())
    price_unit_names_raw = set(df_prices_raw['clean_name'].str.lower().unique())
    matched_before = len(raw_unit_names.intersection(price_unit_names_raw)) - 129 #129 sono quelli che abbiamo messo a mano nel dizionario
    total_units = len(raw_unit_names)
    
    # AFTER normalization
    unit_names_after = set(df_army_units_after['clean_name'].unique())
    price_unit_names_clean = set(df_prices['clean_name'].unique())
    matched_after = len(unit_names_after.intersection(price_unit_names_clean))
    total_after = len(unit_names_after)
    
    results = pd.DataFrame({
        "Stage": ["Before Normalization", "After Normalization"],
        "Unique Unit Names": [total_units, total_after],
        "Matched Names": [matched_before, matched_after],
        "Unmatched Names": [total_units - matched_before, total_after - matched_after],
        "Syntactic Accuracy (%)": [
            (matched_before / total_units) * 100,
            (matched_after / total_after) * 100
        ]
    })
    
    return results

syntactic_accuracy = assess_syntactic_accuracy(df_expanded_before, df_prices_raw, df_army_units_after, df_prices)
print("SYNTACTIC ACCURACY - Unit Name Matching:")
print(syntactic_accuracy.to_string(index=False))

improvement = syntactic_accuracy.iloc[1]['Syntactic Accuracy (%)'] - syntactic_accuracy.iloc[0]['Syntactic Accuracy (%)']
print(f"\n✓ Improvement: +{improvement:.2f}% accuracy")

SYNTACTIC ACCURACY - Unit Name Matching:
               Stage  Unique Unit Names  Matched Names  Unmatched Names  Syntactic Accuracy (%)
Before Normalization                618            432              186               69.902913
 After Normalization                618            618                0              100.000000

✓ Improvement: +30.10% accuracy


### 3. COMPLETENESS (Data Enrichment)

#### Attribute Completeness – Price Enrichment

In [30]:

def assess_completeness_prices(df_expanded_before, df_army_units_after):
    """Assess price data completeness before and after integration"""
    
    total_before = len(df_expanded_before)
    with_price_before = 0  # No prices in raw tournament data
    
    total_after = len(df_army_units_after)
    with_price_after = df_army_units_after['total_price'].notna().sum()
    
    results = pd.DataFrame({
        "Stage": ["Before Integration", "After Integration"],
        "Total Units": [total_before, total_after],
        "Units with Price": [with_price_before, with_price_after],
        "Units without Price": [total_before, total_after - with_price_after],
        "Completeness (%)": [
            (with_price_before / total_before) * 100 if total_before > 0 else 0,
            (with_price_after / total_after) * 100 if total_after > 0 else 0
        ]
    })
    
    return results

completeness_prices = assess_completeness_prices(df_expanded_before, df_army_units_after)
print("COMPLETENESS - Price Data Enrichment:")
print(completeness_prices.to_string(index=False))
print(f"\n✓ {completeness_prices.iloc[1]['Units with Price']} units enriched with prices")

COMPLETENESS - Price Data Enrichment:
             Stage  Total Units  Units with Price  Units without Price  Completeness (%)
Before Integration         4584                 0                 4584               0.0
 After Integration         4584              4584                    0             100.0

✓ 4584 units enriched with prices


#### Schema-level - Missing Values per Table

In [31]:

def assess_completeness_all_tables(conn):
    """Check for missing values across all database tables"""
    
    tables = ["tournaments", "players", "army_lists", "army_list_units", "matches"]
    all_results = []
    
    for table in tables:
        df = pd.read_sql(f"SELECT * FROM {table}", conn)
        
        for col in df.columns:
            missing = df[col].isna().sum()
            total = len(df)
            
            all_results.append({
                "Table": table,
                "Column": col,
                "Missing Values": missing,
                "Total Records": total,
                "Completeness (%)": ((total - missing) / total) * 100
            })
    
    results_df = pd.DataFrame(all_results)
    return results_df

completeness_all = assess_completeness_all_tables(conn)

# Show only columns with missing values
missing_data = completeness_all[completeness_all["Missing Values"] > 0]

if len(missing_data) > 0:
    print("\nCOMPLETENESS - Columns with Missing Values:")
    print(missing_data.to_string(index=False))
else:
    print("\n✓ NO MISSING VALUES in any table!")
    print("\nSample completeness check (first 10 columns):")
    print(completeness_all.head(10)[["Table", "Column", "Completeness (%)"]].to_string(index=False))


✓ NO MISSING VALUES in any table!

Sample completeness check (first 10 columns):
      Table        Column  Completeness (%)
tournaments ID_TOURNAMENT             100.0
tournaments          NAME             100.0
tournaments      LOCATION             100.0
tournaments    EVENT_DATE             100.0
tournaments TOTAL_PLAYERS             100.0
tournaments          YEAR             100.0
    players     ID_PLAYER             100.0
    players ID_TOURNAMENT             100.0
    players          RANK             100.0
    players          NAME             100.0


### 4. CONSISTENCY (Internal Coherence) 

#### Cross-table Consistency (Players vs Matches)

In [32]:

def assess_consistency_match_counts(conn):
    query = """
    SELECT COUNT(*) AS invalid_matches
    FROM matches
    WHERE SCORE_PLAYER1 < SCORE_PLAYER2
    """
    invalid = pd.read_sql(query, conn)['invalid_matches'].iloc[0]

    df_players = pd.read_sql("SELECT WINS, LOSSES, TIES FROM players", conn)
    df_matches = pd.read_sql("SELECT * FROM matches", conn)

    total_matches = len(df_matches)
    total_wins = df_players['WINS'].sum()
    total_ties = df_players['TIES'].sum()
    total_losses = df_players['LOSSES'].sum()

    results = {
        "Invalid Winner Records": invalid,
        "All Winners Valid": invalid == 0,
        "Total Matches (table)": total_matches,
        "Declared Wins": int(total_wins),
        "Declared Ties": int(total_ties),
        "Declared Losses": int(total_losses),
        "Wins + Ties + Losses": int(total_wins + total_ties + total_losses),
        "Consistent": total_matches == (total_wins + total_ties + total_losses)//2
    }

    return results
        
consistency_records = assess_consistency_match_counts(conn)
print("CONSISTENCY - Player Win/Loss Records:")
for key, value in consistency_records.items():
    print(f"  {key}: {value}")

CONSISTENCY - Player Win/Loss Records:
  Invalid Winner Records: 0
  All Winners Valid: True
  Total Matches (table): 1603
  Declared Wins: 1584
  Declared Ties: 38
  Declared Losses: 1584
  Wins + Ties + Losses: 3206
  Consistent: True


#### Domain Consistency - Match Score Validity

In [33]:

def assess_consistency_match_scores(conn):
    """Check if match scores are within valid range (0-100 for Warhammer)"""
    
    query = "SELECT SCORE_PLAYER1, SCORE_PLAYER2 FROM matches"
    df = pd.read_sql(query, conn)
    
    # Warhammer 40K typical score range: 0-100
    invalid = (
        (df['SCORE_PLAYER1'] < 0) | (df['SCORE_PLAYER1'] > 100) |
        (df['SCORE_PLAYER2'] < 0) | (df['SCORE_PLAYER2'] > 100)
    ).sum()
    
    total = len(df)
    
    results = {
        "Total Matches": total,
        "Valid Scores": total - invalid,
        "Invalid Scores": invalid,
        "Score Validity (%)": ((total - invalid) / total) * 100 if total > 0 else 0
    }
    
    return results, df

consistency_scores, df_scores = assess_consistency_match_scores(conn)
print("\nCONSISTENCY - Match Score Validity:")
for key, value in consistency_scores.items():
    print(f"  {key}: {value}")





CONSISTENCY - Match Score Validity:


  Total Matches: 1603
  Valid Scores: 1603
  Invalid Scores: 0
  Score Validity (%): 100.0


#### Domain Consistency - Referential Integrity

In [34]:

def assess_referential_integrity(conn):
    """Check foreign key consistency"""
    
    checks = []
    
    # Check 1: All players in matches exist in players table
    query1 = """
    SELECT COUNT(*) as orphan_count FROM (
        SELECT DISTINCT ID_PLAYER1 as player_id FROM matches
        UNION
        SELECT DISTINCT ID_PLAYER2 FROM matches
    ) m
    WHERE player_id NOT IN (SELECT ID_PLAYER FROM players)
    """
    orphan_players = pd.read_sql(query1, conn)['orphan_count'].iloc[0]
    checks.append(("Players in matches exist", orphan_players == 0))
    
    # Check 2: All players have army lists
    query2 = """
    SELECT COUNT(*) as missing_count 
    FROM players p
    LEFT JOIN army_lists al ON p.ID_PLAYER = al.ID_PLAYER
    WHERE al.ID_PLAYER IS NULL
    """
    missing_lists = pd.read_sql(query2, conn)['missing_count'].iloc[0]
    checks.append(("All players have army lists", missing_lists == 0))
    
    # Check 3: All army_list_units belong to valid players
    query3 = """
    SELECT COUNT(*) as orphan_count
    FROM army_list_units alu
    WHERE alu.ID_PLAYER NOT IN (SELECT ID_PLAYER FROM players)
    """
    orphan_units = pd.read_sql(query3, conn)['orphan_count'].iloc[0]
    checks.append(("All units belong to valid players", orphan_units == 0))
    
    results = pd.DataFrame(checks, columns=["Check", "Passed"])
    results["Status"] = results["Passed"].apply(lambda x: "✓ PASS" if x else "✗ FAIL")
    
    return results

referential_integrity = assess_referential_integrity(conn)
print("\nCONSISTENCY - Referential Integrity:")
print(referential_integrity[["Check", "Status"]].to_string(index=False))


CONSISTENCY - Referential Integrity:
                            Check Status
         Players in matches exist ✓ PASS
      All players have army lists ✓ PASS
All units belong to valid players ✓ PASS


### 5. VALIDITY (Semantic Correctness)

In [35]:

# Checks if data values make sense in the domain context.

def assess_validity_prices(conn, df_prices):
    """Check if calculated prices are semantically valid"""
    
    query = """
    SELECT 
        clean_name,
        points,
        single_price,
        total_price,
        quantity
    FROM army_list_units
    WHERE total_price IS NOT NULL
    """
    df = pd.read_sql(query, conn)
    
    # Check 1: No negative prices
    negative_prices = (df['single_price'] < 0).sum()
    
    # Check 2: Total price = single price * quantity
    df['calculated_total'] = df['single_price'] * df['quantity']
    price_calc_correct = (np.abs(df['total_price'] - df['calculated_total']) < 0.01).sum()
    
    # Check 3: Prices are reasonable (not too high)
    unreasonable_prices = (df['single_price'] > 20000).sum()  # arbitrary threshold
    
    # Check 4: Points/Price ratio is reasonable
    df_merged = df.merge(df_prices, on='clean_name', how='left')
    df_merged['ratio'] = df_merged['points'] / df_merged['single_price']
    reasonable_ratios = df_merged['ratio'].between(0.5, 10).sum()
    total_with_ratio = len(df_merged.dropna(subset=['ratio']))
    
    results = {
        "Total Priced Units": len(df),
        "Negative Prices": negative_prices,
        "Correct Price Calculations": price_calc_correct,
        "Unreasonable Prices (>20000)": unreasonable_prices,
        "Reasonable Point Ratios": f"{reasonable_ratios}/{total_with_ratio}",
        "Price Validity (%)": ((len(df) - negative_prices - unreasonable_prices) / len(df)) * 100 if len(df) > 0 else 0
    }
    
    return results

validity_prices = assess_validity_prices(conn, df_prices)
print("VALIDITY - Price Semantic Correctness:")
for key, value in validity_prices.items():
    print(f"  {key}: {value}")

# %%
def assess_validity_dates(conn):
    """Check if tournament dates are valid"""
    
    query = "SELECT EVENT_DATE, YEAR FROM tournaments"
    df = pd.read_sql(query, conn)
    
    current_year = datetime.now().year
    
    # Convert to datetime if string
    df['EVENT_DATE'] = pd.to_datetime(df['EVENT_DATE'])
    
    future_dates = (df['EVENT_DATE'] > pd.Timestamp.now()).sum()
    too_old = (df['EVENT_DATE'].dt.year < 2020).sum()  # Warhammer 10th edition started 2023
    
    results = {
        "Total Tournaments": len(df),
        "Future Dates": future_dates,
        "Pre-2020 Dates": too_old,
        "Valid Dates": len(df) - future_dates - too_old,
        "Date Validity (%)": ((len(df) - future_dates - too_old) / len(df)) * 100 if len(df) > 0 else 0
    }
    
    return results

validity_dates = assess_validity_dates(conn)
print("\n TEMPORAL VALIDITY - Tournament Dates:")
for key, value in validity_dates.items():
    print(f"  {key}: {value}")

VALIDITY - Price Semantic Correctness:
  Total Priced Units: 4584
  Negative Prices: 0
  Correct Price Calculations: 4584
  Unreasonable Prices (>20000): 0
  Reasonable Point Ratios: 4584/4584
  Price Validity (%): 100.0

 TEMPORAL VALIDITY - Tournament Dates:
  Total Tournaments: 1
  Future Dates: 0
  Pre-2020 Dates: 0
  Valid Dates: 1
  Date Validity (%): 100.0


### SUMMARY DASHBOARD

In [36]:
def build_quality_summary(
    redundancy_prices,
    uniqueness_matches,
    syntactic_accuracy,
    completeness_prices,
    consistency_records,
    consistency_scores,
    referential_integrity,
    validity_prices,
    validity_dates
):
    """
    Build a compact Before/After data quality summary table
    """

    summary = [
        {
            "Dimension": "Uniqueness (Prices)",
            "Metric": "Duplicate reference records",
            "Before (%)": 100 - redundancy_prices.iloc[0]["Redundancy Rate (%)"],
            "After (%)": 100 - redundancy_prices.iloc[1]["Redundancy Rate (%)"]
        },
        {
            "Dimension": "Uniqueness (Matches)",
            "Metric": "Duplicate match records",
            "Before (%)": (1 - uniqueness_matches.iloc[0]["Redundancy Rate"]) * 100,
            "After (%)": (1 - uniqueness_matches.iloc[1]["Redundancy Rate"]) * 100
        },
        {
            "Dimension": "Accuracy",
            "Metric": "Unit name matching",
            "Before (%)": syntactic_accuracy.iloc[0]["Syntactic Accuracy (%)"],
            "After (%)": syntactic_accuracy.iloc[1]["Syntactic Accuracy (%)"]
        },
        {
            "Dimension": "Completeness",
            "Metric": "Price enrichment",
            "Before (%)": completeness_prices.iloc[0]["Completeness (%)"],
            "After (%)": completeness_prices.iloc[1]["Completeness (%)"]
        },
        {
            "Dimension": "Consistency",
            "Metric": "Cross-table logical coherence",
            "Before (%)": 0.0,
            "After (%)": (
                consistency_records["Consistent"]
                and consistency_scores["Score Validity (%)"] == 100.0
                and all(referential_integrity["Passed"])
            ) * 100
        },
        {
            "Dimension": "Validity",
            "Metric": "Semantic & temporal validity",
            "Before (%)": 0.0,
            "After (%)": min(
                validity_prices["Price Validity (%)"],
                validity_dates["Date Validity (%)"]
            )
        }
    ]

    df_summary = pd.DataFrame(summary)
    df_summary["Improvement (%)"] = df_summary["After (%)"] - df_summary["Before (%)"]

    return df_summary


### Results

In [37]:
quality_summary = build_quality_summary(
    redundancy_prices,
    uniqueness_matches,
    syntactic_accuracy,
    completeness_prices,
    consistency_records,
    consistency_scores,
    referential_integrity,
    validity_prices,
    validity_dates
)

quality_summary


,Dimension,Metric,Before (%),After (%),Improvement (%)
0,Uniqueness (Prices),Duplicate reference records,60.317460,100.0,39.682540
1,Uniqueness (Matches),Duplicate match records,50.000000,100.0,50.000000
2,Accuracy,Unit name matching,69.902913,100.0,30.097087
3,Completeness,Price enrichment,0.000000,100.0,100.000000
4,Consistency,Cross-table logical coherence,0.000000,100.0,100.000000
5,Validity,Semantic & temporal validity,0.000000,100.0,100.000000
